# Train 6 ML Models on Clinical Trials Dataset
This notebook loads the dataset, performs exploratory data analysis (EDA), preprocesses the data, trains 6 separate machine learning models (including a highly tuned XGBoost model), and visualizes the results.

## 1. Load Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score

import warnings
warnings.filterwarnings('ignore')

# We assume clinical_trials.csv is in the dataset folder relative to this notebook
df = pd.read_csv('dataset/clinical_trials.csv')
display(df.head())

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Correlation Heatmap of numerical features
numeric_cols = df.select_dtypes(include=[np.number]).columns
plt.figure(figsize=(10, 8))
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Heatmap')
plt.show()

# Distribution of the target variable
plt.figure(figsize=(6, 4))
sns.countplot(x='adverse_event_flag', data=df)
plt.title('Distribution of Adverse Event Flag')
plt.show()


## 3. Preprocess Data

In [ ]:
# We will predict 'adverse_event_flag'
target = 'adverse_event_flag'

# Drop columns that are IDs, hashes or non-predictive
drop_cols = ['trial_id', 'patient_id', 'consent_hash', 'timestamp', 'node_id', 'adverse_event']
X = df.drop(columns=[col for col in drop_cols if col in df.columns] + [target])
y = df[target]

feature_names = X.columns.tolist()

# Encode categorical variables
cat_cols = X.select_dtypes(include=['object']).columns
for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))

# Train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=feature_names)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=feature_names)

print(f"Training set shape: {X_train_scaled.shape}")
print(f"Testing set shape: {X_test_scaled.shape}")


## 4. Initialize Models (with Hyperparameter Tuning for XGBoost)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

# We use GridSearchCV on XGBoost to ensure it achieves high results
xgb_param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [4, 6, 8],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1.0]
}

models = {
    'Logistic Regression': LogisticRegression(),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42),
    'SVM': SVC(probability=True, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Decision Tree': DecisionTreeClassifier(max_depth=10, random_state=42),
    'XGBoost (Tuned)': GridSearchCV(
        XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
        param_grid=xgb_param_grid,
        cv=3,
        scoring='f1',
        n_jobs=-1
    )
}

results = {}


## 5. Train Models and Plot Results

### Model: Logistic Regression

In [ ]:
model = models['Logistic Regression']
print(f"Training { 'Logistic Regression' }...")
model.fit(X_train_scaled, y_train)

if 'Logistic Regression' == 'XGBoost (Tuned)':
    print("Best XGBoost parameters found: ", model.best_params_)

# Predictions
y_pred = model.predict(X_test_scaled)
if hasattr(model, "predict_proba"):
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
else:
    y_prob = model.decision_function(X_test_scaled)

# Metrics
acc = accuracy_score(y_test, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='binary', zero_division=0)
results['Logistic Regression'] = {'Accuracy': acc, 'Precision': precision, 'Recall': recall, 'F1': f1}

print(f"Accuracy: {acc:.4f}")
print(classification_report(y_test, y_pred, zero_division=0))

# Visualizations
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title(f'{'Logistic Regression'} - Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC = {roc_auc:.2f}')
axes[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title(f'{'Logistic Regression'} - ROC Curve')
axes[1].legend(loc="lower right")

# Precision-Recall Curve
precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_prob)
pr_auc = average_precision_score(y_test, y_prob)
axes[2].plot(recall_curve, precision_curve, color='purple', lw=2, label=f'Avg Precision = {pr_auc:.2f}')
axes[2].set_xlabel('Recall')
axes[2].set_ylabel('Precision')
axes[2].set_title(f'{'Logistic Regression'} - Precision-Recall Curve')
axes[2].legend(loc="lower left")

plt.tight_layout()
plt.show()

# Feature Importance
best_model = model.best_estimator_ if hasattr(model, 'best_estimator_') else model
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    indices = np.argsort(importances)[::-1]
    
    plt.figure(figsize=(10, 4))
    plt.title(f"{'Logistic Regression'} - Feature Importances")
    plt.bar(range(X_train_scaled.shape[1]), importances[indices], align="center")
    plt.xticks(range(X_train_scaled.shape[1]), [feature_names[i] for i in indices], rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
elif hasattr(best_model, 'coef_') and 'Logistic Regression' == 'Logistic Regression':
    importances = np.abs(best_model.coef_[0])
    indices = np.argsort(importances)[::-1]
    
    plt.figure(figsize=(10, 4))
    plt.title(f"{'Logistic Regression'} - Feature Importances (Absolute Coefficients)")
    plt.bar(range(X_train_scaled.shape[1]), importances[indices], align="center")
    plt.xticks(range(X_train_scaled.shape[1]), [feature_names[i] for i in indices], rotation=45, ha='right')
    plt.tight_layout()
    plt.show()


### Model: Random Forest

In [ ]:
model = models['Random Forest']
print(f"Training { 'Random Forest' }...")
model.fit(X_train_scaled, y_train)

if 'Random Forest' == 'XGBoost (Tuned)':
    print("Best XGBoost parameters found: ", model.best_params_)

# Predictions
y_pred = model.predict(X_test_scaled)
if hasattr(model, "predict_proba"):
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
else:
    y_prob = model.decision_function(X_test_scaled)

# Metrics
acc = accuracy_score(y_test, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='binary', zero_division=0)
results['Random Forest'] = {'Accuracy': acc, 'Precision': precision, 'Recall': recall, 'F1': f1}

print(f"Accuracy: {acc:.4f}")
print(classification_report(y_test, y_pred, zero_division=0))

# Visualizations
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title(f'{'Random Forest'} - Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC = {roc_auc:.2f}')
axes[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title(f'{'Random Forest'} - ROC Curve')
axes[1].legend(loc="lower right")

# Precision-Recall Curve
precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_prob)
pr_auc = average_precision_score(y_test, y_prob)
axes[2].plot(recall_curve, precision_curve, color='purple', lw=2, label=f'Avg Precision = {pr_auc:.2f}')
axes[2].set_xlabel('Recall')
axes[2].set_ylabel('Precision')
axes[2].set_title(f'{'Random Forest'} - Precision-Recall Curve')
axes[2].legend(loc="lower left")

plt.tight_layout()
plt.show()

# Feature Importance
best_model = model.best_estimator_ if hasattr(model, 'best_estimator_') else model
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    indices = np.argsort(importances)[::-1]
    
    plt.figure(figsize=(10, 4))
    plt.title(f"{'Random Forest'} - Feature Importances")
    plt.bar(range(X_train_scaled.shape[1]), importances[indices], align="center")
    plt.xticks(range(X_train_scaled.shape[1]), [feature_names[i] for i in indices], rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
elif hasattr(best_model, 'coef_') and 'Random Forest' == 'Logistic Regression':
    importances = np.abs(best_model.coef_[0])
    indices = np.argsort(importances)[::-1]
    
    plt.figure(figsize=(10, 4))
    plt.title(f"{'Random Forest'} - Feature Importances (Absolute Coefficients)")
    plt.bar(range(X_train_scaled.shape[1]), importances[indices], align="center")
    plt.xticks(range(X_train_scaled.shape[1]), [feature_names[i] for i in indices], rotation=45, ha='right')
    plt.tight_layout()
    plt.show()


### Model: SVM

In [ ]:
model = models['SVM']
print(f"Training { 'SVM' }...")
model.fit(X_train_scaled, y_train)

if 'SVM' == 'XGBoost (Tuned)':
    print("Best XGBoost parameters found: ", model.best_params_)

# Predictions
y_pred = model.predict(X_test_scaled)
if hasattr(model, "predict_proba"):
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
else:
    y_prob = model.decision_function(X_test_scaled)

# Metrics
acc = accuracy_score(y_test, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='binary', zero_division=0)
results['SVM'] = {'Accuracy': acc, 'Precision': precision, 'Recall': recall, 'F1': f1}

print(f"Accuracy: {acc:.4f}")
print(classification_report(y_test, y_pred, zero_division=0))

# Visualizations
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title(f'{'SVM'} - Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC = {roc_auc:.2f}')
axes[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title(f'{'SVM'} - ROC Curve')
axes[1].legend(loc="lower right")

# Precision-Recall Curve
precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_prob)
pr_auc = average_precision_score(y_test, y_prob)
axes[2].plot(recall_curve, precision_curve, color='purple', lw=2, label=f'Avg Precision = {pr_auc:.2f}')
axes[2].set_xlabel('Recall')
axes[2].set_ylabel('Precision')
axes[2].set_title(f'{'SVM'} - Precision-Recall Curve')
axes[2].legend(loc="lower left")

plt.tight_layout()
plt.show()

# Feature Importance
best_model = model.best_estimator_ if hasattr(model, 'best_estimator_') else model
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    indices = np.argsort(importances)[::-1]
    
    plt.figure(figsize=(10, 4))
    plt.title(f"{'SVM'} - Feature Importances")
    plt.bar(range(X_train_scaled.shape[1]), importances[indices], align="center")
    plt.xticks(range(X_train_scaled.shape[1]), [feature_names[i] for i in indices], rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
elif hasattr(best_model, 'coef_') and 'SVM' == 'Logistic Regression':
    importances = np.abs(best_model.coef_[0])
    indices = np.argsort(importances)[::-1]
    
    plt.figure(figsize=(10, 4))
    plt.title(f"{'SVM'} - Feature Importances (Absolute Coefficients)")
    plt.bar(range(X_train_scaled.shape[1]), importances[indices], align="center")
    plt.xticks(range(X_train_scaled.shape[1]), [feature_names[i] for i in indices], rotation=45, ha='right')
    plt.tight_layout()
    plt.show()


### Model: KNN

In [ ]:
model = models['KNN']
print(f"Training { 'KNN' }...")
model.fit(X_train_scaled, y_train)

if 'KNN' == 'XGBoost (Tuned)':
    print("Best XGBoost parameters found: ", model.best_params_)

# Predictions
y_pred = model.predict(X_test_scaled)
if hasattr(model, "predict_proba"):
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
else:
    y_prob = model.decision_function(X_test_scaled)

# Metrics
acc = accuracy_score(y_test, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='binary', zero_division=0)
results['KNN'] = {'Accuracy': acc, 'Precision': precision, 'Recall': recall, 'F1': f1}

print(f"Accuracy: {acc:.4f}")
print(classification_report(y_test, y_pred, zero_division=0))

# Visualizations
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title(f'{'KNN'} - Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC = {roc_auc:.2f}')
axes[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title(f'{'KNN'} - ROC Curve')
axes[1].legend(loc="lower right")

# Precision-Recall Curve
precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_prob)
pr_auc = average_precision_score(y_test, y_prob)
axes[2].plot(recall_curve, precision_curve, color='purple', lw=2, label=f'Avg Precision = {pr_auc:.2f}')
axes[2].set_xlabel('Recall')
axes[2].set_ylabel('Precision')
axes[2].set_title(f'{'KNN'} - Precision-Recall Curve')
axes[2].legend(loc="lower left")

plt.tight_layout()
plt.show()

# Feature Importance
best_model = model.best_estimator_ if hasattr(model, 'best_estimator_') else model
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    indices = np.argsort(importances)[::-1]
    
    plt.figure(figsize=(10, 4))
    plt.title(f"{'KNN'} - Feature Importances")
    plt.bar(range(X_train_scaled.shape[1]), importances[indices], align="center")
    plt.xticks(range(X_train_scaled.shape[1]), [feature_names[i] for i in indices], rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
elif hasattr(best_model, 'coef_') and 'KNN' == 'Logistic Regression':
    importances = np.abs(best_model.coef_[0])
    indices = np.argsort(importances)[::-1]
    
    plt.figure(figsize=(10, 4))
    plt.title(f"{'KNN'} - Feature Importances (Absolute Coefficients)")
    plt.bar(range(X_train_scaled.shape[1]), importances[indices], align="center")
    plt.xticks(range(X_train_scaled.shape[1]), [feature_names[i] for i in indices], rotation=45, ha='right')
    plt.tight_layout()
    plt.show()


### Model: Decision Tree

In [ ]:
model = models['Decision Tree']
print(f"Training { 'Decision Tree' }...")
model.fit(X_train_scaled, y_train)

if 'Decision Tree' == 'XGBoost (Tuned)':
    print("Best XGBoost parameters found: ", model.best_params_)

# Predictions
y_pred = model.predict(X_test_scaled)
if hasattr(model, "predict_proba"):
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
else:
    y_prob = model.decision_function(X_test_scaled)

# Metrics
acc = accuracy_score(y_test, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='binary', zero_division=0)
results['Decision Tree'] = {'Accuracy': acc, 'Precision': precision, 'Recall': recall, 'F1': f1}

print(f"Accuracy: {acc:.4f}")
print(classification_report(y_test, y_pred, zero_division=0))

# Visualizations
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title(f'{'Decision Tree'} - Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC = {roc_auc:.2f}')
axes[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title(f'{'Decision Tree'} - ROC Curve')
axes[1].legend(loc="lower right")

# Precision-Recall Curve
precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_prob)
pr_auc = average_precision_score(y_test, y_prob)
axes[2].plot(recall_curve, precision_curve, color='purple', lw=2, label=f'Avg Precision = {pr_auc:.2f}')
axes[2].set_xlabel('Recall')
axes[2].set_ylabel('Precision')
axes[2].set_title(f'{'Decision Tree'} - Precision-Recall Curve')
axes[2].legend(loc="lower left")

plt.tight_layout()
plt.show()

# Feature Importance
best_model = model.best_estimator_ if hasattr(model, 'best_estimator_') else model
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    indices = np.argsort(importances)[::-1]
    
    plt.figure(figsize=(10, 4))
    plt.title(f"{'Decision Tree'} - Feature Importances")
    plt.bar(range(X_train_scaled.shape[1]), importances[indices], align="center")
    plt.xticks(range(X_train_scaled.shape[1]), [feature_names[i] for i in indices], rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
elif hasattr(best_model, 'coef_') and 'Decision Tree' == 'Logistic Regression':
    importances = np.abs(best_model.coef_[0])
    indices = np.argsort(importances)[::-1]
    
    plt.figure(figsize=(10, 4))
    plt.title(f"{'Decision Tree'} - Feature Importances (Absolute Coefficients)")
    plt.bar(range(X_train_scaled.shape[1]), importances[indices], align="center")
    plt.xticks(range(X_train_scaled.shape[1]), [feature_names[i] for i in indices], rotation=45, ha='right')
    plt.tight_layout()
    plt.show()


### Model: XGBoost (Tuned)

In [ ]:
model = models['XGBoost (Tuned)']
print(f"Training { 'XGBoost (Tuned)' }...")
model.fit(X_train_scaled, y_train)

if 'XGBoost (Tuned)' == 'XGBoost (Tuned)':
    print("Best XGBoost parameters found: ", model.best_params_)

# Predictions
y_pred = model.predict(X_test_scaled)
if hasattr(model, "predict_proba"):
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
else:
    y_prob = model.decision_function(X_test_scaled)

# Metrics
acc = accuracy_score(y_test, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='binary', zero_division=0)
results['XGBoost (Tuned)'] = {'Accuracy': acc, 'Precision': precision, 'Recall': recall, 'F1': f1}

print(f"Accuracy: {acc:.4f}")
print(classification_report(y_test, y_pred, zero_division=0))

# Visualizations
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title(f'{'XGBoost (Tuned)'} - Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC = {roc_auc:.2f}')
axes[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title(f'{'XGBoost (Tuned)'} - ROC Curve')
axes[1].legend(loc="lower right")

# Precision-Recall Curve
precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_prob)
pr_auc = average_precision_score(y_test, y_prob)
axes[2].plot(recall_curve, precision_curve, color='purple', lw=2, label=f'Avg Precision = {pr_auc:.2f}')
axes[2].set_xlabel('Recall')
axes[2].set_ylabel('Precision')
axes[2].set_title(f'{'XGBoost (Tuned)'} - Precision-Recall Curve')
axes[2].legend(loc="lower left")

plt.tight_layout()
plt.show()

# Feature Importance
best_model = model.best_estimator_ if hasattr(model, 'best_estimator_') else model
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    indices = np.argsort(importances)[::-1]
    
    plt.figure(figsize=(10, 4))
    plt.title(f"{'XGBoost (Tuned)'} - Feature Importances")
    plt.bar(range(X_train_scaled.shape[1]), importances[indices], align="center")
    plt.xticks(range(X_train_scaled.shape[1]), [feature_names[i] for i in indices], rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
elif hasattr(best_model, 'coef_') and 'XGBoost (Tuned)' == 'Logistic Regression':
    importances = np.abs(best_model.coef_[0])
    indices = np.argsort(importances)[::-1]
    
    plt.figure(figsize=(10, 4))
    plt.title(f"{'XGBoost (Tuned)'} - Feature Importances (Absolute Coefficients)")
    plt.bar(range(X_train_scaled.shape[1]), importances[indices], align="center")
    plt.xticks(range(X_train_scaled.shape[1]), [feature_names[i] for i in indices], rotation=45, ha='right')
    plt.tight_layout()
    plt.show()


## 6. Compare All Models

In [ ]:
results_df = pd.DataFrame(results).T
display(results_df)

# Plot comparison
results_df.plot(kind='bar', figsize=(14, 6))
plt.title('Model Comparison on Adverse Event Prediction')
plt.ylabel('Score')
plt.xticks(rotation=45)
plt.legend(loc='lower right', bbox_to_anchor=(1.1, 0))
plt.tight_layout()
plt.show()
